In [1]:
import numpy as np


class KMeansFromScratch:
    def __init__(self, k=3, max_iters=100, tol=1e-4, random_state=None):
        self.k = k
        self.max_iters = max_iters
        self.tol = tol
        self.random_state = random_state

        self.centroids = None
        self.labels = None
        self.inertia_ = None

    def _initialize_centroids(self, X):
        if self.random_state is not None:
            np.random.seed(self.random_state)

        random_indices = np.random.choice(X.shape[0], self.k, replace=False)
        return X[random_indices]

    def _compute_distances(self, X, centroids):
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        return distances

    def fit(self, X):
        X = np.asarray(X, dtype=np.float64)

        self.centroids = self._initialize_centroids(X)

        for iteration in range(self.max_iters):

            distances = self._compute_distances(X, self.centroids)
            self.labels = np.argmin(distances, axis=1)

            new_centroids = np.zeros_like(self.centroids)
            for k in range(self.k):
                cluster_points = X[self.labels == k]
                if len(cluster_points) > 0:
                    new_centroids[k] = np.mean(cluster_points, axis=0)
                else:

                    new_centroids[k] = X[np.random.choice(X.shape[0])]
            centroid_shift = np.linalg.norm(self.centroids - new_centroids)
            self.centroids = new_centroids

            if centroid_shift < self.tol:
                break
        final_distances = self._compute_distances(X, self.centroids)
        min_distances = np.min(final_distances, axis=1)
        self.inertia_ = np.sum(min_distances**2)

        return self

    def predict(self, X):
        """Predict the closest cluster for each sample in X."""
        X = np.asarray(X, dtype=np.float64)
        distances = self._compute_distances(X, self.centroids)
        return np.argmin(distances, axis=1)

if __name__ == "__main__":
    from sklearn.datasets import make_blobs
    from sklearn.cluster import KMeans as SklearnKMeans

    # Generate synthetic 2D data
    X, y_true = make_blobs(
        n_samples=300, centers=3, cluster_std=0.60, random_state=42
    )

    # 1. Fit scratch implementation
    custom_kmeans = KMeansFromScratch(k=3, random_state=42)
    custom_kmeans.fit(X)

    # 2. Fit sklearn implementation
    sklearn_kmeans = SklearnKMeans(
        n_clusters=3, n_init=1, init="random", random_state=42
    )
    sklearn_kmeans.fit(X)

    # Display centroids
    print("--- Centroids from Scratch ---")
    print(np.sort(custom_kmeans.centroids, axis=0))

    print("\n--- Centroids from Scikit-Learn ---")
    print(np.sort(sklearn_kmeans.cluster_centers_, axis=0))

    # Display inertia (Within-Cluster Sum of Squares)
    print(f"\nScratch Inertia: {custom_kmeans.inertia_:.4f}")
    print(f"Sklearn Inertia: {sklearn_kmeans.inertia_:.4f}")

--- Centroids from Scratch ---
[[-6.88217395 -6.94243433]
 [-2.58361866  1.99562444]
 [ 4.70421356  9.03185632]]

--- Centroids from Scikit-Learn ---
[[-6.88217395 -6.94243433]
 [-2.58361866  1.99562444]
 [ 4.70421356  9.03185632]]

Scratch Inertia: 204.0694
Sklearn Inertia: 204.0694


C:\Users\Dell\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
